# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Setup: load data, rebuild the label exactly as Week 5 did ----
import os, sys, subprocess
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

RNG = 42
np.random.seed(RNG)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("Loaded:", df.shape)
print("is_declining base rate:", round(df["is_declining"].mean(), 3), "(matches Week 5: 0.542)")



Loaded: (30000, 45)
is_declining base rate: 0.542 (matches Week 5: 0.542)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Re-run the Week-5 model under a naive random split vs. the grouped split ----
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RNG = 42
np.random.seed(RNG)

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def make_logreg():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, random_state=RNG))])

X_all = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_all = df["is_declining"]

print("=== BEFORE: naive random (row-level) split ===")
ss = ShuffleSplit(n_splits=1, test_size=0.3, random_state=RNG)
tr_idx, te_idx = next(ss.split(X_all))
Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
ytr, yte = y_all.iloc[tr_idx], y_all.iloc[te_idx]
overlap_naive = len(set(df.iloc[tr_idx]["client_id"]) & set(df.iloc[te_idx]["client_id"]))
model_naive = make_logreg()
model_naive.fit(Xtr, ytr)
proba_naive = model_naive.predict_proba(Xte)[:, 1]
auc_naive = roc_auc_score(yte, proba_naive)
p50_naive = precision_at_k(yte, proba_naive, 50)
print(f"Client overlap train/test: {overlap_naive} / {df['client_id'].nunique()} total clients (row-level split ignores client boundaries)")
print(f"roc_auc: {auc_naive:.3f} | precision@50: {p50_naive:.3f}")

print("\n=== AFTER: grouped split by client_id (same design as Week 5) ===")
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RNG)
tr_idx2, te_idx2 = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[tr_idx2].copy(), df.iloc[te_idx2].copy()
Xtr2 = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
Xte2 = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
ytr2, yte2 = train_df["is_declining"], test_df["is_declining"]
overlap_grouped = len(set(train_df["client_id"]) & set(test_df["client_id"]))
model_grouped = make_logreg()
model_grouped.fit(Xtr2, ytr2)
proba_grouped = model_grouped.predict_proba(Xte2)[:, 1]
auc_grouped = roc_auc_score(yte2, proba_grouped)
p50_grouped = precision_at_k(yte2, proba_grouped, 50)
print(f"Client overlap train/test: {overlap_grouped}")
print(f"roc_auc: {auc_grouped:.3f} | precision@50: {p50_grouped:.3f}")

print("\n=== Comparison table ===")
comparison = pd.DataFrame([
    {"split": "before: naive random (row-level)", "client_overlap": overlap_naive, "roc_auc": round(auc_naive, 3), "precision@50": round(p50_naive, 3)},
    {"split": "after: grouped by client_id", "client_overlap": overlap_grouped, "roc_auc": round(auc_grouped, 3), "precision@50": round(p50_grouped, 3)},
])
print(comparison.to_string(index=False))
print(f"\nGap: roc_auc -{auc_naive - auc_grouped:.3f} | precision@50 -{p50_naive - p50_grouped:.3f} once client overlap is removed.")
print("This grouped number (0.621 / 0.70) matches what was already reported in Week 5 -- confirming Week 5's split was already honest.")


=== BEFORE: naive random (row-level) split ===
Client overlap train/test: 32 / 32 total clients (row-level split ignores client boundaries)
roc_auc: 0.716 | precision@50: 0.880

=== AFTER: grouped split by client_id (same design as Week 5) ===
Client overlap train/test: 0
roc_auc: 0.621 | precision@50: 0.700

=== Comparison table ===
                           split  client_overlap  roc_auc  precision@50
before: naive random (row-level)              32    0.716          0.88
     after: grouped by client_id               0    0.621          0.70

Gap: roc_auc -0.095 | precision@50 -0.180 once client overlap is removed.
This grouped number (0.621 / 0.70) matches what was already reported in Week 5 -- confirming Week 5's split was already honest.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Leakage hunt: add each previously-excluded column back one at a time ----
SUSPECTS = ["trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

def fit_eval(numeric_extra):
    num_feats = NUMERIC_FEATURES + numeric_extra
    Xtr = train_df[num_feats + CATEGORICAL_FEATURES]
    Xte = test_df[num_feats + CATEGORICAL_FEATURES]
    ytr, yte = train_df["is_declining"], test_df["is_declining"]
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_feats),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                           ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
    ])
    pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, random_state=RNG))])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, proba), pipe

print("=== Leakage hunt: add back each excluded column one at a time, grouped split, same fold ===")
baseline_auc, model_grouped_final = fit_eval([])
print(f"WITHOUT suspects (final Week-5 feature set): roc_auc = {baseline_auc:.3f}")
for s in SUSPECTS:
    auc_with, _ = fit_eval([s])
    print(f"WITH '{s}' added: roc_auc = {auc_with:.3f}  (jump: {auc_with - baseline_auc:+.3f})")

df["computed_trend_pct"] = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].replace(0, np.nan)
corr = df[["trend_pct", "computed_trend_pct"]].corr().iloc[0, 1]
print(f"\nConfession check: correlation of trend_pct with (impressions_last_30d - impressions_prev_30d)/impressions_prev_30d = {corr:.3f}")
print("is_declining is literally the sign of trend_pct -- this is textbook label-derived leakage (taxonomy #1), not a borderline case.")

# ---- Real failure examples on the grouped-split model, final safe feature set ----
print("\n=== Real failure examples (grouped-split model, final safe feature set) ===")
Xte_final = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
proba_final = model_grouped_final.predict_proba(Xte_final)[:, 1]
test_df = test_df.assign(pred_proba=proba_final)

fp = test_df[(test_df["is_declining"] == 0)].sort_values("pred_proba", ascending=False).head(3)
fn = test_df[(test_df["is_declining"] == 1)].sort_values("pred_proba", ascending=True).head(3)

cols_show = ["content_id", "content_type", "main_intent", "avg_position", "days_with_impressions",
             "content_age_days", "freshness_tier", "pred_proba", "is_declining"]
print("\nFalse positives (model was confident it's declining, but it wasn't):")
print(fp[cols_show].to_string(index=False))
print("\nFalse negatives (model was confident it's fine, but it was actually declining):")
print(fn[cols_show].to_string(index=False))



=== Leakage hunt: add back each excluded column one at a time, grouped split, same fold ===
WITHOUT suspects (final Week-5 feature set): roc_auc = 0.621
WITH 'trend_pct' added: roc_auc = 0.989  (jump: +0.369)
WITH 'impressions_last_30d' added: roc_auc = 0.670  (jump: +0.050)
WITH 'clicks_last_30d' added: roc_auc = 0.623  (jump: +0.002)
WITH 'sessions_last_30d' added: roc_auc = 0.627  (jump: +0.006)
WITH 'impressions_prev_30d' added: roc_auc = 0.622  (jump: +0.001)
WITH 'clicks_prev_30d' added: roc_auc = 0.621  (jump: +0.000)
WITH 'sessions_prev_30d' added: roc_auc = 0.621  (jump: +0.000)

Confession check: correlation of trend_pct with (impressions_last_30d - impressions_prev_30d)/impressions_prev_30d = 1.000
is_declining is literally the sign of trend_pct -- this is textbook label-derived leakage (taxonomy #1), not a borderline case.

=== Real failure examples (grouped-split model, final safe feature set) ===

False positives (model was confident it's declining, but it wasn't):
      

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Sanity-check the numbers the claim rewrite below leans on ----
print(f"precision@50 on the grouped (honest) split: {p50_grouped:.3f}")
print(f"-> of the top 50 pages the model ranked highest-risk, {round(p50_grouped * 50)} of 50 were, in this dataset, observed to be declining.")
print(f"roc_auc on the grouped split: {auc_grouped:.3f} (base rate of is_declining: {df['is_declining'].mean():.3f})")



precision@50 on the grouped (honest) split: 0.700
-> of the top 50 pages the model ranked highest-risk, 35 of 50 were, in this dataset, observed to be declining.
roc_auc on the grouped split: 0.621 (base rate of is_declining: 0.542)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.